# DBTS Train-Only Diagnostic Experiment — fixed mechanical version

This notebook is intentionally **TRAIN ONLY**.

- Fit models using only `split.train_idx`.
- Run DBTS dynamic target selection using only `split.train_idx`.
- Do not use validation dates.
- Do not use test dates.

This version executes every non-flat classifier signal mechanically, without PositionManager blocking, so it can diagnose whether DBTS is alive: target switches, selected targets, trades, Sharpe, trade log, and confusion matrix.


In [14]:
# Cell 1 — TRAIN-ONLY safety switch
TRAIN_ONLY_MODE = True
ALLOW_ANY_NON_TRAIN_DATA = False

assert TRAIN_ONLY_MODE is True
assert ALLOW_ANY_NON_TRAIN_DATA is False

print("TRAIN-ONLY MODE ACTIVE")
print("TRAIN used for fitting: YES")
print("TRAIN used for DBTS diagnostics/evaluation: YES")
print("Any non-train data used: NO")


TRAIN-ONLY MODE ACTIVE
TRAIN used for fitting: YES
TRAIN used for DBTS diagnostics/evaluation: YES
Any non-train data used: NO


In [15]:
# Cell 2 — imports and project path setup
import os
import sys
import math
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore")
%matplotlib inline

# Adjust this if your project lives elsewhere.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "strategy").exists():
    candidate = Path(r"C:\algo-trading-project")
    if candidate.exists():
        PROJECT_ROOT = candidate

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")


Project root: c:\algo-trading-project


In [16]:
# Cell 3 — project imports
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split, walk_forward_folds
from strategy.predictor_selector import PredictorSelector
from strategy.regressors import DynamicShadowPriceModel, DynamicReturnModel
from strategy.residual_features import ResidualFeatureBuilder
from strategy.technical_features import TechnicalRuleFeatureBuilder
from strategy.bandit_target_selector import BanditTargetSelector
from strategy.classifier import GlobalSignalClassifier, make_labels

print("Project imports loaded.")


Project imports loaded.


In [17]:
# Cell 4 — helpers

def safe_price(prices: pd.DataFrame, ticker: str, date):
    try:
        v = float(prices.at[date, ticker])
        return v if math.isfinite(v) and v > 0 else float("nan")
    except Exception:
        return float("nan")


def make_actual_label(price_series: pd.Series, cfg) -> pd.Series:
    """Use the project's label function for consistency."""
    return make_labels(price_series, cfg)


def signal_name(x):
    return {-1: "short", 0: "flat", 1: "long"}.get(int(x), str(x)) if pd.notna(x) else "nan"


def calc_trade_return(signal, entry_price, exit_price):
    if not (math.isfinite(entry_price) and math.isfinite(exit_price)):
        return float("nan")
    if entry_price <= 0 or exit_price <= 0:
        return float("nan")
    if int(signal) == 1:     # long
        return exit_price / entry_price - 1.0
    if int(signal) == -1:    # short
        return entry_price / exit_price - 1.0
    return 0.0              # flat / no trade


def max_drawdown(equity: pd.Series):
    if equity.empty:
        return float("nan")
    peak = equity.cummax()
    dd = equity / peak - 1.0
    return float(dd.min())


def perf_metrics(trades: pd.DataFrame, periods_per_year=252):
    if trades.empty:
        return pd.Series(dtype=float)
    executed = trades[trades["executed"] == True].copy()
    returns = executed["realized_return"].dropna()
    if returns.empty:
        return pd.Series({
            "trades": 0, "cumulative_return": 0.0, "annualized_return": np.nan,
            "annualized_volatility": np.nan, "sharpe": np.nan, "sortino": np.nan,
            "max_drawdown": np.nan, "win_rate": np.nan, "avg_trade_return": np.nan,
            "median_trade_return": np.nan, "long_trades": 0, "short_trades": 0,
            "flat_predictions": int((trades["signal"] == 0).sum())
        })
    equity = (1.0 + returns).cumprod()
    cum = float(equity.iloc[-1] - 1.0)
    ann_ret = float((1.0 + cum) ** (periods_per_year / max(len(returns), 1)) - 1.0)
    ann_vol = float(returns.std(ddof=1) * np.sqrt(periods_per_year)) if len(returns) > 1 else np.nan
    sharpe = float(ann_ret / ann_vol) if ann_vol and np.isfinite(ann_vol) and ann_vol != 0 else np.nan
    downside = returns[returns < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(periods_per_year)) if len(downside) > 1 else np.nan
    sortino = float(ann_ret / downside_vol) if downside_vol and np.isfinite(downside_vol) and downside_vol != 0 else np.nan
    return pd.Series({
        "trades": int(len(returns)),
        "cumulative_return": cum,
        "annualized_return": ann_ret,
        "annualized_volatility": ann_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown(equity),
        "win_rate": float((returns > 0).mean()),
        "avg_trade_return": float(returns.mean()),
        "median_trade_return": float(returns.median()),
        "long_trades": int((executed["signal"] == 1).sum()),
        "short_trades": int((executed["signal"] == -1).sum()),
        "flat_predictions": int((trades["signal"] == 0).sum())
    })


In [18]:
# Cell 5 — load data and define TRAIN ONLY index
# This cell creates exactly one active date index: train_idx.
# No validation/test index is created, counted, printed, or used.

cfg = StrategyConfig(force_recompute=True, make_plots=False)
pipeline = StrategyPipeline(cfg)

print("Loading market data...")
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)

train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
horizon_buffer = int(max(getattr(cfg, "label_horizon", 1), getattr(cfg, "return_horizon", 1)))

# Dates that can safely have forward labels/exits without looking outside train.
train_fit_idx = train_idx[:-horizon_buffer] if len(train_idx) > horizon_buffer else train_idx[:0]
train_run_idx = train_idx[:-horizon_buffer] if len(train_idx) > horizon_buffer else train_idx[:0]

print("ACTIVE EXPERIMENT SPLIT USAGE:")
print(f"TRAIN used for fitting: YES | {train_idx[0].date()} -> {train_idx[-1].date()} | n={len(train_idx)}")
print(f"TRAIN safe fit/eval dates: {train_fit_idx[0].date()} -> {train_fit_idx[-1].date()} | n={len(train_fit_idx)}")
print("Any non-train dates used: NO")
print(f"Forward horizon buffer removed from train tail: {horizon_buffer} trading days")

assert len(train_idx) > 0
assert len(train_fit_idx) > 0
assert train_fit_idx.max() <= train_idx.max()
assert train_run_idx.max() <= train_idx.max()


Loading market data...
[cache] FORCE market_data__0296fb90c88f.pkl -> computing
[fetch] 120 tickers | 2021-01-01 -> today | 1d
[INFO] 203 returns exceed 15% — kept
[load] Ready — 1358 trading days x 120 stocks
ACTIVE EXPERIMENT SPLIT USAGE:
TRAIN used for fitting: YES | 2021-01-04 -> 2024-04-01 | n=815
TRAIN safe fit/eval dates: 2021-01-04 -> 2024-03-28 | n=814
Any non-train dates used: NO
Forward horizon buffer removed from train tail: 1 trading days


In [19]:
# Cell 6 — build a TRAIN-only panel and fit classifier using TRAIN rows only
# Walk-forward folds are generated only inside train_fit_idx.
# The last horizon_buffer train days are removed so labels/exits never look outside TRAIN.

train_folds = walk_forward_folds(train_fit_idx, cfg)
print(f"TRAIN-only folds: {len(train_folds)}")
for i, f in enumerate(train_folds, 1):
    print(f"  fold {i:02d}: train={len(f.train_idx):4d} | predict={len(f.predict_idx):3d} | retrain={f.retrain_date.date()}")

print("Building TRAIN-only feature panel...")
panel = pipeline.build_panel(md, train_folds, split)
panel["date"] = pd.to_datetime(panel["date"])
panel = panel[panel["date"].isin(train_fit_idx)].copy()
print(f"Panel rows after strict TRAIN-only filter: {len(panel):,}")

# Prepare classifier data manually so we do not call pipeline.train_classifier,
# because that method contains project-level dev/test split logic.
excluded = (
    "date", "etf", "sector", "target", "predictors", "target_price",
    "shadow_price", "next_ret", "label", "spread_signal",
    "ann_vol", "residual_z", "price_residual", "residual_ewm_mean",
    "residual_ewm_std", "residual_roll_mean", "residual_roll_std"
)
feature_cols = [c for c in panel.columns if c not in excluded]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(panel[c])]

if not feature_cols:
    raise ValueError("No numeric feature columns found in TRAIN-only panel.")

data = panel.dropna(subset=["label"]).copy()
X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.groupby(data["target"]).ffill().fillna(0.0)
for c in feature_cols:
    data[c] = X[c]

data = data.sort_values("date")
# Internal classifier check split is inside TRAIN only. No external split is used.
cut_pos = max(1, int(len(data) * 0.80))
train_fit_rows = data.iloc[:cut_pos].copy()
train_check_rows = data.iloc[cut_pos:].copy()
if train_check_rows.empty:
    train_check_rows = train_fit_rows.copy()

clf = GlobalSignalClassifier(cfg)
print("Fitting classifier on TRAIN-only data...")
clf_result = clf.fit(
    train_fit_rows[feature_cols], train_fit_rows["label"],
    train_check_rows[feature_cols], train_check_rows["label"],
)

print("Classifier fit complete.")
print("Rows used:")
print({
    "train_fit_rows": len(train_fit_rows),
    "train_internal_check_rows": len(train_check_rows),
    "total_train_panel_rows": len(data),
})
print("Training label distribution:")
display(data["label"].value_counts().sort_index().rename(index={-1:"short",0:"flat",1:"long"}).to_frame("count"))
print("Top 20 feature importances:")
display(clf_result.feature_importance.head(20).to_frame("importance"))


TRAIN-only folds: 2
  fold 01: train= 756 | predict= 50 | retrain=2024-01-05
  fold 02: train= 806 | predict=  8 | retrain=2024-03-19
Building TRAIN-only feature panel...
[cache] FORCE feature_panel__e194d6d00390.pkl -> computing
[cache] FORCE technical_all__8647065ddef7.pkl -> computing

  OOS modelling — Materials (XLB)
[target-select] stage=candidate_frame sector=Materials retrain_date=2024-01-05 candidates=11
[target-select]   sector=Materials candidate=FCX status=score_start completed_candidates=0
[target-select]   sector=Materials candidate=FCX status=score_done completed_candidates=1
[target-select]   sector=Materials candidate=SCCO status=score_start completed_candidates=1
[target-select]   sector=Materials candidate=SCCO status=score_done completed_candidates=2
[target-select]   sector=Materials candidate=NEM status=score_start completed_candidates=2
[target-select]   sector=Materials candidate=NEM status=score_done completed_candidates=3
[target-select]   sector=Materials can

,count
label,
short,97
flat,359
long,124


Top 20 feature importances:


,importance
oh_XLK,0.034437
rule_near_52w_high,0.028321
rule_sma50_gt_200,0.024218
oh_XLC,0.023430
rule_obv_trend_pos,0.023145
rule_obv_trend_neg,0.022509
oh_XLB,0.021170
residual_sign,0.020312
sect_vol_rel_etf,0.019524
px_ema200_ratio_z,0.019073


In [20]:
# Cell 7 — fit per-sector/per-candidate shadow and return models on TRAIN only
print("Fitting per-sector candidate models on TRAIN only...")

model_store = {}
predictor_rows = []
bandit = BanditTargetSelector(cfg)
completed = 0

for etf, cfg_sector in SECTORS.items():
    sector_name = cfg_sector["name"]
    members = [cfg_sector["target"]] + cfg_sector["predictors"]
    model_store[etf] = {}
    print(f"[fit] sector={sector_name}, candidates={len(members)}")

    for cand in members:
        peers = [m for m in members if m != cand and m in md.prices.columns]
        if cand not in md.prices.columns or not peers:
            predictor_rows.append({"sector": sector_name, "candidate": cand, "status": "skipped", "predictors_used": ""})
            continue

        psel = PredictorSelector(cfg)
        pred_choice = psel.select(cand, peers, md.returns.reindex(train_fit_idx), md.prices.loc[train_fit_idx])
        preds = list(pred_choice.selected)

        # Safety check: candidate must not be among predictors.
        assert cand not in preds, f"Leakage: {cand} appears in its own predictors"

        shadow_m = DynamicShadowPriceModel(cfg)
        shadow_feats, _, base_price, safe_idx = shadow_m.fit(md.prices, cand, preds, train_fit_idx)

        return_m = DynamicReturnModel(cfg)
        return_feats, _, _ = return_m.fit(md.prices, cand, preds, train_idx)

        model_store[etf][cand] = dict(
            predictors=preds,
            shadow_model=shadow_m,
            shadow_feats=shadow_feats,
            base_price=base_price,
            return_model=return_m,
            return_feats=return_feats,
        )
        predictor_rows.append({
            "sector": sector_name,
            "candidate": cand,
            "status": "fit_done",
            "n_predictors": len(preds),
            "predictors_used": ",".join(preds),
        })
        completed += 1
        print(f"[fit]   candidate={cand}, predictors={len(preds)}, completed={completed}")

    bandit.init_sector(sector_name, members)

predictor_summary = pd.DataFrame(predictor_rows)
display(predictor_summary)
print(f"Completed candidate models: {completed}")


Fitting per-sector candidate models on TRAIN only...
[fit] sector=Materials, candidates=11
[fit]   candidate=FCX, predictors=5, completed=1
[fit]   candidate=SCCO, predictors=5, completed=2
[fit]   candidate=NEM, predictors=5, completed=3
[fit]   candidate=AA, predictors=5, completed=4
[fit]   candidate=CLF, predictors=5, completed=5
[fit]   candidate=NUE, predictors=5, completed=6
[fit]   candidate=VMC, predictors=3, completed=7
[fit]   candidate=MLM, predictors=5, completed=8
[fit]   candidate=ALB, predictors=5, completed=9
[fit]   candidate=SQM, predictors=5, completed=10
[fit]   candidate=TECK, predictors=5, completed=11
[fit] sector=Communication, candidates=11
[fit]   candidate=META, predictors=5, completed=12
[fit]   candidate=GOOGL, predictors=5, completed=13
[fit]   candidate=PINS, predictors=5, completed=14
[fit]   candidate=SNAP, predictors=5, completed=15
[fit]   candidate=TTD, predictors=5, completed=16
[fit]   candidate=NFLX, predictors=5, completed=17
[fit]   candidate=D

,sector,candidate,status,n_predictors,predictors_used
0,Materials,FCX,fit_done,5,"SCCO,TECK,AA,CLF,NUE"
1,Materials,SCCO,fit_done,5,"FCX,AA,TECK,NEM,SQM"
2,Materials,NEM,fit_done,5,"SCCO,AA,FCX,TECK,MLM"
3,Materials,AA,fit_done,5,"FCX,SCCO,TECK,NUE,CLF"
4,Materials,CLF,fit_done,5,"NUE,FCX,AA,SQM,TECK"
...,...,...,...,...,...
105,Consumer Disc.,MCD,fit_done,3,"SBUX,MAR,HD"
106,Consumer Disc.,BKNG,fit_done,5,"EXPE,MAR,SBUX,AMZN,NKE"
107,Consumer Disc.,EXPE,fit_done,5,"BKNG,MAR,MCD,LOW,TSLA"
108,Consumer Disc.,MAR,fit_done,5,"EXPE,BKNG,MCD,LOW,LULU"


Completed candidate models: 110


In [ ]:
# Cell 8 — cached DBTS on TRAIN only, robust mechanical diagnostic
# First run = heavy. Later runs = load from cache.
# Uses ONLY train_idx. No validation. No test.

from pathlib import Path

CACHE_DIR = Path("outputs/train_only_dbts_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRADES_FILE = CACHE_DIR / "trades.pkl"
SCORES_FILE = CACHE_DIR / "daily_scores.pkl"
BANDIT_FILE = CACHE_DIR / "bandit_states.pkl"

RECOMPUTE_DBTS = False  # change to True only if you want to force rerun

if (
    not RECOMPUTE_DBTS
    and TRADES_FILE.exists()
    and SCORES_FILE.exists()
    and BANDIT_FILE.exists()
):
    print("Loaded cached DBTS TRAIN-only results.")
    trades = pd.read_pickle(TRADES_FILE)
    daily_scores = pd.read_pickle(SCORES_FILE)
    bandit_states = pd.read_pickle(BANDIT_FILE)

else:
    print("Running ROBUST DBTS dynamic target selection on TRAIN only...")

    tech_builder = TechnicalRuleFeatureBuilder(cfg)
    resid_builder = ResidualFeatureBuilder(cfg)

    trade_rows = []
    daily_score_rows = []
    bandit_state_rows = []

    DBTS_WEIGHTS = {"bandit": 0.35, "residual": 0.45, "adf": 0.20}
    MIN_HISTORY_FOR_DBTS = max(90, int(getattr(cfg, "min_train_days", 252) // 2))
    ADF_RECOMPUTE_EVERY = 20

    h = int(getattr(cfg, "label_horizon", 5))
    eval_dates = list(train_idx[MIN_HISTORY_FOR_DBTS:-h]) if h > 0 else list(train_idx[MIN_HISTORY_FOR_DBTS:])

    print(
        f"TRAIN-only DBTS dates: {len(eval_dates)} | "
        f"from {pd.Timestamp(eval_dates[0]).date()} to {pd.Timestamp(eval_dates[-1]).date()}"
    )

    assert len(eval_dates) > 0
    assert max(eval_dates) <= train_idx[-1]

    adf_cache = {}
    last_selected_by_sector = {}

    def _normalize_signal(raw_signal):
        try:
            s = int(raw_signal)
        except Exception:
            return 0
        if s in (-1, 0, 1):
            return s
        if s in (0, 1, 2):
            return {0: -1, 1: 0, 2: 1}[s]
        return 0

    def _fallback_residual_z(price_series: pd.Series, date, window=60):
        hist = price_series.loc[:date].dropna().astype(float)
        hist = hist[hist > 0]
        if len(hist) < max(20, window // 2):
            return np.nan
        tail = hist.tail(window)
        mu = tail.mean()
        sd = tail.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return np.nan
        return float((hist.iloc[-1] - mu) / sd)

    def _get_classifier_prediction(feat_row, date):
        X_row = pd.DataFrame([{c: feat_row.get(c, 0.0) for c in clf.features_}], index=[date])
        X_row = (
            X_row
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )
        proba = clf.predict_proba(X_row)
        p_short, p_flat, p_long = [float(x) for x in proba.iloc[0].tolist()]
        raw_signal = clf.predict(X_row).iloc[0]
        signal = _normalize_signal(raw_signal)
        return signal, p_short, p_flat, p_long

    train_idx_list = list(train_idx)

    for date_no, date in enumerate(eval_dates, start=1):
        if date_no == 1 or date_no % 25 == 0:
            print(f"[dbts-train] date {date_no}/{len(eval_dates)}: {pd.Timestamp(date).date()}")

        for etf, cfg_sector in SECTORS.items():
            sector_name = cfg_sector["name"]
            members = [cfg_sector["target"]] + cfg_sector["predictors"]
            bandit_samples = bandit.sample_scores(sector_name)

            scores = {}
            component_map = {}

            for cand in members:
                rec = model_store.get(etf, {}).get(cand)

                if rec is None or cand not in md.prices.columns:
                    scores[cand] = -np.inf
                    component_map[cand] = dict(
                        residual_z=np.nan,
                        residual_score=0.0,
                        adf_pvalue=np.nan,
                        adf_score=0.0,
                        bandit_score=bandit_samples.get(cand, np.nan),
                        valid_candidate=False,
                    )
                    continue

                feats = rec["shadow_feats"]
                predict_idx = feats.loc[:date].index if not feats.empty else pd.Index([])
                predict_idx = pd.DatetimeIndex([d for d in predict_idx if d in train_idx and d <= date])

                residual_z = np.nan
                adf_p = np.nan
                resid_series = pd.Series(dtype=float)

                if len(predict_idx) > 0:
                    try:
                        shadow_pred = rec["shadow_model"].predict(feats, predict_idx, rec["base_price"])
                        pred_ret = rec["return_model"].predict(rec["return_feats"], predict_idx)
                        price_hist = md.prices[cand].reindex(predict_idx)
                        resid_series = (price_hist - shadow_pred).dropna()

                        rf = resid_builder.build(price_hist, shadow_pred, pred_ret)
                        if date in rf.index and "residual_ewm_z" in rf.columns:
                            residual_z = float(rf.at[date, "residual_ewm_z"])
                    except Exception:
                        residual_z = np.nan

                if not np.isfinite(residual_z):
                    residual_z = _fallback_residual_z(md.prices[cand], date, window=60)

                residual_score = min(abs(residual_z) / 3.0, 1.0) if np.isfinite(residual_z) else 0.0

                cache_key = (sector_name, cand)
                should_recompute = (cache_key not in adf_cache) or (date_no % ADF_RECOMPUTE_EVERY == 0)

                if should_recompute:
                    try:
                        from statsmodels.tsa.stattools import adfuller

                        if len(resid_series.dropna()) >= 30:
                            adf_p = float(adfuller(resid_series.dropna(), autolag="AIC")[1])
                        else:
                            px = md.prices[cand].loc[:date].dropna().astype(float)
                            px = px[px > 0].tail(120)
                            dev = px - px.rolling(30, min_periods=10).mean()
                            dev = dev.dropna()
                            adf_p = float(adfuller(dev, autolag="AIC")[1]) if len(dev) >= 30 else np.nan
                    except Exception:
                        adf_p = np.nan

                    adf_cache[cache_key] = adf_p
                else:
                    adf_p = adf_cache.get(cache_key, np.nan)

                adf_score = 1.0 - min(adf_p, 1.0) if np.isfinite(adf_p) else 0.0
                bandit_score = float(bandit_samples.get(cand, 0.5))

                final_score = (
                    DBTS_WEIGHTS["bandit"] * bandit_score
                    + DBTS_WEIGHTS["residual"] * residual_score
                    + DBTS_WEIGHTS["adf"] * adf_score
                )

                scores[cand] = final_score
                component_map[cand] = dict(
                    residual_z=residual_z,
                    residual_score=residual_score,
                    adf_pvalue=adf_p,
                    adf_score=adf_score,
                    bandit_score=bandit_score,
                    valid_candidate=True,
                )

            for cand in members:
                comp = component_map.get(cand, {})
                daily_score_rows.append({
                    "date": date,
                    "sector": sector_name,
                    "candidate": cand,
                    "valid_candidate": comp.get("valid_candidate", False),
                    "bandit_score": comp.get("bandit_score", np.nan),
                    "residual_z": comp.get("residual_z", np.nan),
                    "residual_score": comp.get("residual_score", np.nan),
                    "adf_pvalue": comp.get("adf_pvalue", np.nan),
                    "adf_score": comp.get("adf_score", np.nan),
                    "final_score": scores.get(cand, np.nan),
                })

            finite_scores = {k: v for k, v in scores.items() if np.isfinite(v)}
            selected = max(finite_scores.items(), key=lambda kv: kv[1])[0] if finite_scores else members[0]

            target_switched = selected != last_selected_by_sector.get(sector_name, selected)
            last_selected_by_sector[sector_name] = selected

            rec = model_store.get(etf, {}).get(selected)
            selected_comp = component_map.get(selected, {})

            feat_row = {}

            try:
                tech = tech_builder.build(selected, md.prices, md.highs, md.lows, md.volumes)
                if date in tech.index:
                    for c, v in tech.loc[date].items():
                        feat_row[c] = float(v) if pd.notna(v) and np.isfinite(v) else 0.0
            except Exception:
                pass

            if rec is not None:
                try:
                    feats = rec["shadow_feats"]
                    predict_idx = feats.loc[:date].index if not feats.empty else pd.Index([])
                    predict_idx = pd.DatetimeIndex([d for d in predict_idx if d in train_idx and d <= date])

                    if len(predict_idx):
                        shadow_pred = rec["shadow_model"].predict(feats, predict_idx, rec["base_price"])
                        pred_ret = rec["return_model"].predict(rec["return_feats"], predict_idx)
                        rf = resid_builder.build(md.prices[selected].reindex(predict_idx), shadow_pred, pred_ret)

                        if date in rf.index:
                            for c in rf.columns:
                                v = rf.at[date, c]
                                feat_row[c] = float(v) if pd.notna(v) and np.isfinite(v) else 0.0
                except Exception:
                    pass

            signal, p_short, p_flat, p_long = _get_classifier_prediction(feat_row, date)

            try:
                label_series = make_actual_label(md.prices[selected], cfg)
                actual_label = label_series.get(date, np.nan)
            except Exception:
                actual_label = np.nan

            try:
                i = train_idx_list.index(date)
                exit_date = train_idx_list[i + h] if i + h < len(train_idx_list) else pd.NaT
            except Exception:
                exit_date = pd.NaT

            entry_price = safe_price(md.prices, selected, date)
            exit_price = safe_price(md.prices, selected, exit_date) if pd.notna(exit_date) else np.nan

            executed = signal in (-1, 1) and math.isfinite(entry_price) and math.isfinite(exit_price)
            realized = calc_trade_return(signal, entry_price, exit_price) if executed else 0.0

            alpha_before, beta_before = bandit.get_state(sector_name, selected)

            if executed:
                a0, b0, updated = bandit.update(sector_name, selected, realized)
            else:
                a0, b0, updated = alpha_before, beta_before, "no_trade_no_update"

            alpha_after, beta_after = bandit.get_state(sector_name, selected)

            bandit_state_rows.append({
                "date": date,
                "sector": sector_name,
                "selected_target": selected,
                "alpha_before": alpha_before,
                "beta_before": beta_before,
                "alpha_after": alpha_after,
                "beta_after": beta_after,
                "bandit_updated": updated,
            })

            trade_rows.append({
                "date": date,
                "sector": sector_name,
                "selected_target": selected,
                "target_switched": bool(target_switched),
                "predictors_used": ",".join(rec["predictors"]) if rec else "",
                "signal": int(signal),
                "direction": signal_name(signal),
                "actual_label": actual_label,
                "P_short": p_short,
                "P_flat": p_flat,
                "P_long": p_long,
                "residual_z": selected_comp.get("residual_z", np.nan),
                "adf_pvalue": selected_comp.get("adf_pvalue", np.nan),
                "bandit_score": selected_comp.get("bandit_score", np.nan),
                "residual_score": selected_comp.get("residual_score", np.nan),
                "adf_score": selected_comp.get("adf_score", np.nan),
                "final_target_score": scores.get(selected, np.nan),
                "entry_date": date,
                "exit_date": exit_date,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "holding_period": h,
                "executed": bool(executed),
                "realized_return": realized,
                "alpha_before": alpha_before,
                "beta_before": beta_before,
                "alpha_after": alpha_after,
                "beta_after": beta_after,
                "bandit_updated": updated,
                "reason_for_entry": "non_flat_classifier_signal" if executed else "flat_signal_or_missing_price",
                "reason_for_exit": f"fixed_horizon_{h}_days",
            })

    trades = pd.DataFrame(trade_rows)
    daily_scores = pd.DataFrame(daily_score_rows)
    bandit_states = pd.DataFrame(bandit_state_rows)

    trades.to_pickle(TRADES_FILE)
    daily_scores.to_pickle(SCORES_FILE)
    bandit_states.to_pickle(BANDIT_FILE)

    print("Saved DBTS TRAIN-only cache files.")

print("DBTS TRAIN-only run ready.")
print(f"Rows in trade log: {len(trades):,}")
print(f"Rows in daily scores: {len(daily_scores):,}")
print("Direction counts:")
display(trades["direction"].value_counts().to_frame("count"))
print("Executed trades:", int(trades["executed"].sum()))
print("Target switches:", int(trades["target_switched"].sum()))
display(trades.head())

Running ROBUST DBTS dynamic target selection on TRAIN only...
TRAIN-only DBTS dates: 436 | from 2022-07-06 to 2024-03-28
[dbts-train] date 1/436: 2022-07-06


In [ ]:
# Cell 9 — overall performance metrics
metrics = perf_metrics(trades)
display(metrics.to_frame("value"))

print("Signal distribution:")
display(trades["direction"].value_counts().to_frame("count"))

print("Executed trade count:")
print(int(trades["executed"].sum()))


In [ ]:
# Cell 10 — dynamic target selection diagnostics
# Number of target switches by sector.
target_switch_rows = []
for sector, sub in trades.sort_values("date").groupby("sector"):
    switches = int((sub["selected_target"] != sub["selected_target"].shift(1)).sum() - 1) if len(sub) > 1 else 0
    target_switch_rows.append({
        "sector": sector,
        "target_switches": max(switches, 0),
        "most_selected_target": sub["selected_target"].mode().iloc[0] if not sub.empty else None,
        "unique_targets_selected": int(sub["selected_target"].nunique()),
        "rows": len(sub),
    })
target_switch_summary = pd.DataFrame(target_switch_rows).sort_values("target_switches", ascending=False)
display(target_switch_summary)

selection_summary = (
    trades.groupby(["sector", "selected_target"])
    .agg(
        times_selected=("selected_target", "size"),
        avg_return=("realized_return", "mean"),
        avg_final_score=("final_target_score", "mean"),
        avg_bandit_score=("bandit_score", "mean"),
        avg_residual_score=("residual_score", "mean"),
        avg_adf_score=("adf_score", "mean"),
        avg_abs_residual_z=("residual_z", lambda x: np.nanmean(np.abs(x))),
        executed_trades=("executed", "sum"),
    )
    .reset_index()
)
selection_summary["selection_pct_in_sector"] = selection_summary["times_selected"] / selection_summary.groupby("sector")["times_selected"].transform("sum")
display(selection_summary.sort_values(["sector", "times_selected"], ascending=[True, False]))


In [ ]:
# Cell 11 — sector performance
sector_summary = (
    trades.groupby("sector")
    .apply(lambda g: perf_metrics(g))
    .reset_index()
)
sector_summary = sector_summary.merge(target_switch_summary, on="sector", how="left")
display(sector_summary.sort_values("sharpe", ascending=False))


In [ ]:
# Cell 12 — full trade log
# Display a compact version first. The full CSV is saved later.
cols = [
    "date", "sector", "selected_target", "direction", "executed", "P_short", "P_flat", "P_long",
    "residual_z", "adf_pvalue", "bandit_score", "residual_score", "adf_score", "final_target_score",
    "entry_price", "exit_price", "holding_period", "realized_return", "alpha_before", "beta_before", "alpha_after", "beta_after"
]
display(trades[cols].head(100))


In [ ]:
# Cell 13 — confusion matrix and classification report on TRAIN selected-target rows
cm_df = pd.DataFrame()
report_df = pd.DataFrame()

clf_rows = trades.dropna(subset=["actual_label", "signal"]).copy()
if not clf_rows.empty:
    labels = [-1, 0, 1]
    cm = confusion_matrix(clf_rows["actual_label"].astype(int), clf_rows["signal"].astype(int), labels=labels)
    cm_df = pd.DataFrame(cm, index=["actual_short", "actual_flat", "actual_long"], columns=["pred_short", "pred_flat", "pred_long"])
    display(cm_df)
    report = classification_report(clf_rows["actual_label"].astype(int), clf_rows["signal"].astype(int), labels=labels, target_names=["short", "flat", "long"], output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    display(report_df)
else:
    print("No rows available for confusion matrix.")


In [ ]:
# Cell 14 — plots: equity curve and drawdown
executed = trades[trades["executed"] == True].copy().sort_values("date")
if not executed.empty:
    executed["equity"] = (1.0 + executed["realized_return"].fillna(0)).cumprod()
    executed["drawdown"] = executed["equity"] / executed["equity"].cummax() - 1.0

    plt.figure(figsize=(14, 5))
    plt.plot(executed["date"], executed["equity"])
    plt.title("TRAIN-only DBTS Equity Curve")
    plt.xlabel("Date")
    plt.ylabel("Equity")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(14, 5))
    plt.plot(executed["date"], executed["drawdown"])
    plt.title("TRAIN-only DBTS Drawdown")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No executed trades to plot equity/drawdown.")


In [ ]:
# Cell 15 — plots: target selection frequency
if not selection_summary.empty:
    for sector, sub in selection_summary.groupby("sector"):
        sub = sub.sort_values("times_selected", ascending=False)
        plt.figure(figsize=(12, 4))
        plt.bar(sub["selected_target"], sub["times_selected"])
        plt.title(f"Target selection frequency — {sector}")
        plt.xlabel("Selected target")
        plt.ylabel("Times selected")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:
# Cell 16 — plots: returns and PnL by sector/target
if not executed.empty:
    plt.figure(figsize=(10, 5))
    plt.hist(executed["realized_return"].dropna(), bins=40)
    plt.title("Executed trade return distribution")
    plt.xlabel("Trade return")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    pnl_sector = executed.groupby("sector")["realized_return"].sum().sort_values(ascending=False)
    plt.figure(figsize=(12, 5))
    plt.bar(pnl_sector.index, pnl_sector.values)
    plt.title("Cumulative simple PnL by sector — TRAIN only")
    plt.xlabel("Sector")
    plt.ylabel("Sum of trade returns")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    pnl_target = executed.groupby("selected_target")["realized_return"].sum().sort_values(ascending=False).head(30)
    plt.figure(figsize=(14, 5))
    plt.bar(pnl_target.index, pnl_target.values)
    plt.title("Top cumulative simple PnL by selected target — TRAIN only")
    plt.xlabel("Selected target")
    plt.ylabel("Sum of trade returns")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No executed trades for PnL plots.")


In [ ]:
# Cell 17 — bandit diagnostics
if not bandit_states.empty:
    bandit_update_summary = (
        bandit_states.groupby(["sector", "selected_target", "bandit_updated"])
        .size().unstack(fill_value=0).reset_index()
    )
    display(bandit_update_summary)

    # Plot alpha/beta evolution for the most selected target in each sector.
    for sector, sub in bandit_states.groupby("sector"):
        top_target = sub["selected_target"].value_counts().idxmax()
        ss = sub[sub["selected_target"] == top_target].sort_values("date")
        plt.figure(figsize=(12, 4))
        plt.plot(ss["date"], ss["alpha_after"], label="alpha")
        plt.plot(ss["date"], ss["beta_after"], label="beta")
        plt.title(f"Bandit alpha/beta evolution — {sector} / {top_target}")
        plt.xlabel("Date")
        plt.ylabel("Value")
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


In [ ]:
# Cell 18 — sanity checks
sanity = []

sanity.append(("Any non-train data used", "PASS" if not ALLOW_ANY_NON_TRAIN_DATA else "FAIL"))
sanity.append(("Only train_idx used for fit/eval", "PASS" if TRAIN_ONLY_MODE else "FAIL"))
sanity.append(("Train tail horizon removed", "PASS" if len(train_fit_idx) < len(train_idx) else "FAIL"))

# Check selected target excluded from predictors.
if not trades.empty and "predictors_used" in trades:
    bad_pred = trades.apply(lambda r: str(r["selected_target"]) in str(r["predictors_used"]).split(","), axis=1).sum()
    sanity.append(("Selected target excluded from predictors", "PASS" if bad_pred == 0 else f"FAIL ({bad_pred})"))
else:
    sanity.append(("Selected target excluded from predictors", "UNKNOWN"))

nan_feature_rows = int(trades[["P_short", "P_flat", "P_long", "final_target_score"]].isna().any(axis=1).sum()) if not trades.empty else 0
sanity.append(("No NaN leakage in key outputs", "PASS" if nan_feature_rows == 0 else f"WARN ({nan_feature_rows} rows)"))

# Short return correctness sample check.
shorts = trades[(trades["executed"] == True) & (trades["signal"] == -1)].copy()
if not shorts.empty:
    expected_short = shorts["entry_price"] / shorts["exit_price"] - 1.0
    diff = (expected_short - shorts["realized_return"]).abs().max()
    sanity.append(("Short return calculation correct", "PASS" if diff < 1e-10 else f"FAIL max_diff={diff}"))
else:
    sanity.append(("Short return calculation correct", "NO SHORTS TO CHECK"))

sanity_df = pd.DataFrame(sanity, columns=["check", "status"])
display(sanity_df)


In [ ]:
# Cell 19 — save TRAIN-only outputs
out_dir = PROJECT_ROOT / "outputs"
out_dir.mkdir(exist_ok=True)

trades.to_csv(out_dir / "dbts_train_trade_log.csv", index=False)
daily_scores.to_csv(out_dir / "dbts_train_daily_target_scores.csv", index=False)
sector_summary.to_csv(out_dir / "dbts_train_sector_summary.csv", index=False)
selection_summary.to_csv(out_dir / "dbts_train_target_selection_summary.csv", index=False)
cm_df.to_csv(out_dir / "dbts_train_confusion_matrix.csv", index=True)
metrics.to_frame("value").to_csv(out_dir / "dbts_train_metrics.csv")
predictor_summary.to_csv(out_dir / "dbts_train_predictor_summary.csv", index=False)
sanity_df.to_csv(out_dir / "dbts_train_sanity_checks.csv", index=False)

print("Saved TRAIN-only DBTS outputs:")
for name in [
    "dbts_train_trade_log.csv",
    "dbts_train_daily_target_scores.csv",
    "dbts_train_sector_summary.csv",
    "dbts_train_target_selection_summary.csv",
    "dbts_train_confusion_matrix.csv",
    "dbts_train_metrics.csv",
    "dbts_train_predictor_summary.csv",
    "dbts_train_sanity_checks.csv",
]:
    print(out_dir / name)


In [ ]:
# Cell 20 — final plain-English conclusion helper
print("TRAIN-ONLY DBTS diagnostic complete.")
print("Use these points to judge whether the mechanism is alive:")
print(f"- Executed trades: {int(trades['executed'].sum()) if not trades.empty else 0}")
print(f"- Unique selected targets: {int(trades['selected_target'].nunique()) if not trades.empty else 0}")
print(f"- Total target switches across sectors: {int(target_switch_summary['target_switches'].sum()) if not target_switch_summary.empty else 0}")
print(f"- Sharpe: {metrics.get('sharpe', np.nan)}")
print(f"- Cumulative return: {metrics.get('cumulative_return', np.nan)}")
print("Reminder: this is IN-SAMPLE TRAIN ONLY. It checks mechanics, not out-of-sample performance.")
